In [17]:
import sys
# !{sys.executable} -m pip install tiktoken
# !{sys.executable} -m pip install pandas>=2.0.0
import os

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # for 1B
import gc
gc.collect()
torch.cuda.empty_cache()

Using device: cuda


In [ ]:
import json
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch_dtype = torch.float16
attn_implementation = "eager"
# !{sys.executable} -m pip install bitsandbytes
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)


In [ ]:

base_model_id = "Qwen/Qwen2.5-7B"
fine_tuned_dir = "xfu20/BEMGPT_1.0.0"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    fine_tuned_dir,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/srv/data1/fuxiaoqin/conda_envs/py7/lib/python3.10/site-packages/transformers/quantizers/auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.53it/s]


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import requests

In [22]:
def download_github_json_files(repo_owner, repo_name, folder_path, branch="main"):
    """
    Recursively downloads all JSON (.json) files from a GitHub repository folder
    to the current directory.
    
    Parameters:
    - repo_owner: GitHub username or organization
    - repo_name: repository name
    - folder_path: path to folder in the repo
    - branch: branch name (default "main")
    """
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{folder_path}?ref={branch}"
    r = requests.get(api_url)
    if r.status_code != 200:
        print(f"Failed to access {api_url}: {r.status_code}")
        return

    contents = r.json()
    
    for item in contents:
        item_type = item['type']
        item_name = item['name']
        item_path = item['path']
        if item_type == "file" and item_name.endswith(".json"):
            download_url = item['download_url']
            print(f"Downloading {item_path}...")
            file_resp = requests.get(download_url)
            file_path = os.path.join(os.getcwd(), item_name)  # save in current directory
            with open(file_path, "wb") as f:
                f.write(file_resp.content)
        elif item_type == "dir":
            # Recursive call for subdirectories
            download_github_json_files(repo_owner, repo_name, item_path, branch)

repo_owner = "fuArizona"
repo_name = "BEMGPT"
folder_path = "data/dataset/training/Pairs"
download_github_json_files(repo_owner, repo_name, folder_path)

In [23]:
# ----------------------------------
# List of potential QA pair files
# ----------------------------------
qa_files = [
    "engineering-reference.json",
]

# ----------------------------------
# Load QA pairs from all existing files
# ----------------------------------
qa_pairs = []
for path in qa_files:
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    qa_pairs.extend(data)
        except Exception as e:
            print(f"Failed to load {path}: {e}")


# ----------------------------------
# Handle empty or missing QA data
# ----------------------------------
if qa_pairs:
    questions = [pair.get("input", "").strip() for pair in qa_pairs if pair.get("input")]
    answers = [pair.get("output", "").strip() for pair in qa_pairs if pair.get("output")]
else:
    qa_pairs, questions, answers = [], [], []

# ----------------------------------
# Build FAISS retrieval index
# ----------------------------------
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
if questions:
    question_embeddings = embed_model.encode(questions, convert_to_numpy=True)
    embedding_dim = question_embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)  # Inner product = cosine when normalized
    faiss.normalize_L2(question_embeddings)
    index.add(question_embeddings)
else:
    # Empty fallback index (zero-dimension) for safe querying
    index = faiss.IndexFlatIP(384)  # default dim for MiniLM

/srv/data1/fuxiaoqin/conda_envs/py7/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [24]:
def generate_with_model(prompt, model, tokenizer, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.5,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [25]:
def ask(query, model, tokenizer, top_k=3, max_new_tokens=512):

    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)

    threshold = 0.8
    base_answer = None
    if D[0][0] >= threshold:
        candidate_answer = answers[I[0][0]]
        if candidate_answer and candidate_answer.strip():
            base_answer = candidate_answer.strip()


    if base_answer:
        prompt = f"""
You are an expert in building energy simulation and EnergyPlus.
Focus only on answering the following question using the context of the original answer.
Do not include unrelated information or excessive elaboration.

**Question:** {query}

**Original Answer:** {base_answer}

**Expanded and Polished Answer (aligned with the EnergyPlus Engineering Reference):**
"""
    else:
        prompt = f"""
You are an expert in building energy simulation and EnergyPlus.
Answer the following question clearly, concisely, and accurately.
Only include information relevant to the question.

**Question:** {query}

**Answer (aligned with the EnergyPlus Engineering Reference):**
"""


    polished_answer = generate_with_model(prompt, model, tokenizer, max_new_tokens=max_new_tokens)

    # Clean output: keep only text after the headers
    marker1 = "**Expanded and Polished Answer (aligned with the EnergyPlus Engineering Reference):**"
    marker2 = "**Answer (aligned with the EnergyPlus Engineering Reference):**"
    
    if marker1 in polished_answer:
        polished_answer = polished_answer.split(marker1)[-1].strip()
    elif marker2 in polished_answer:
        polished_answer = polished_answer.split(marker2)[-1].strip()

    polished_answer = polished_answer.replace(' ```', '')
    # Ensure the text ends with a proper sentence
    if not polished_answer.endswith(('.', '!', '?')):
        polished_answer += '.'

    return polished_answer


In [26]:
response = ask("In EnergyPlus, what are two main types of loops within the HVAC simulation?", model, tokenizer)
print(response)

In EnergyPlus, the two primary types of loops involved in the HVAC simulation are the Air Loop and the Plant Loop. These loops facilitate the distribution of conditioned air and the production/transfer of thermal energy, respectively. The Air Loop manages air handling units and associated ductwork, while the Plant Loop handles chillers, boilers, and other equipment that manage water-based systems for heating and cooling. Both loops work together to ensure efficient and effective building climate control.


In [27]:
response = ask("In EnergyPlus, what is the equation to calculate infiltration according to Coblenz and Achenbach in 1963?", model, tokenizer)
print(response)

In EnergyPlus, the infiltration calculation based on Coblenz and Achenbach's 1963 model is given by:
Infiltration = (I_(design))(F_(schedule))[A + B|T_(zone)-T_(odb)| + C(V_1) + D(V_1^2)]
where:
- I_(design) is the design infiltration rate per square meter,
- F_(schedule) is the schedule value for infiltration,
- T_(zone) is the zone air temperature,
- T_(odb) is the outdoor air dry-bulb temperature,
- V_1 is the wind speed at the height of the building’s reference point. 

This equation models infiltration as a function of design parameters, schedules, temperature differences between the zone and outdoors, and the effects of wind speed on infiltration rates. The coefficients A, B, C, and D are determined from empirical data and specific to the building being modeled.
